# Notebook 06 — Captura del mapa interactivo (PNG + GIF)

Usamos **Playwright** (Chromium headless) para:
1. Cargar el HTML generado por el notebook 05.
2. Esperar a que las tiles y el NDVI se rendericen.
3. Capturar un PNG estático.
4. Capturar una secuencia de ~25 frames con zoom/pan lento para componer un GIF demostrativo.

**Salidas**: `media/05_mapa_interactivo.png`, `media/05_mapa_interactivo.gif`

In [1]:
from pathlib import Path
import os
import asyncio
import time
from PIL import Image
from playwright.async_api import async_playwright

def find_taller_root(start: Path, marker: str = "python/data/bogota_sabana_sentinel2.tif") -> Path:
    p = start.resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró {marker} desde {start}")

ROOT = find_taller_root(Path.cwd())
os.chdir(ROOT)
MEDIA = ROOT / "media"
MEDIA.mkdir(exist_ok=True)

HTML_PATH = MEDIA / "05_mapa_interactivo.html"
PNG_PATH = MEDIA / "05_mapa_interactivo.png"
GIF_PATH = MEDIA / "05_mapa_interactivo.gif"
print("HTML:", HTML_PATH)
print("PNG destino:", PNG_PATH)
print("GIF destino:", GIF_PATH)

HTML: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/05_mapa_interactivo.html
PNG destino: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/05_mapa_interactivo.png
GIF destino: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/05_mapa_interactivo.gif


In [2]:
async def _capture():
    frames = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context(
            viewport={"width": 1200, "height": 800},
            device_scale_factor=1,
        )
        page = await context.new_page()
        await page.goto(f"file://{HTML_PATH.resolve()}")
        await page.wait_for_load_state("networkidle")
        await page.wait_for_timeout(4500)

        # PNG estático inicial
        await page.screenshot(path=str(PNG_PATH), full_page=False)
        print(f"  PNG estático guardado: {PNG_PATH}")

        # Localizar el contenedor del mapa (Leaflet)
        map_box = await page.locator(".leaflet-container").first.bounding_box()
        if map_box is None:
            raise RuntimeError("No se encontró .leaflet-container")
        cx = map_box["x"] + map_box["width"] / 2
        cy = map_box["y"] + map_box["height"] / 2

        # Botón de zoom-in (+) y zoom-out (-)
        zoom_in = page.locator("a.leaflet-control-zoom-in")
        zoom_out = page.locator("a.leaflet-control-zoom-out")

        # Secuencia: 3 zoom-in (z=10 → z=13), 4 drag pan, 3 zoom-out
        steps = []
        for _ in range(3):
            steps.append(("zoom_in",))
        for dx, dy in [(120, 0), (-180, 80), (140, -90), (-100, -50)]:
            steps.append(("drag", dx, dy))
        for _ in range(3):
            steps.append(("zoom_out",))
        # volver a zoom_in x2 para terminar cerca
        for _ in range(2):
            steps.append(("zoom_in",))

        for i, step in enumerate(steps):
            if step[0] == "zoom_in":
                await zoom_in.click()
            elif step[0] == "zoom_out":
                await zoom_out.click()
            elif step[0] == "drag":
                _, dx, dy = step
                await page.mouse.move(cx, cy)
                await page.mouse.down()
                steps_n = 8
                for s in range(1, steps_n + 1):
                    await page.mouse.move(cx + dx * s / steps_n, cy + dy * s / steps_n)
                    await page.wait_for_timeout(40)
                await page.mouse.up()
            await page.wait_for_timeout(900)
            from io import BytesIO
            buf = await page.screenshot(full_page=False)
            img = Image.open(BytesIO(buf)).convert("RGB").resize((720, 480))
            frames.append(img)
            print(f"  frame {i+1}/{len(steps)} ({step[0]})")

        await browser.close()
    return frames

frames = await _capture()
print(f"\n{len(frames)} frames capturados")

  PNG estático guardado: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/05_mapa_interactivo.png


  frame 1/12 (zoom_in)


  frame 2/12 (zoom_in)


  frame 3/12 (zoom_in)


  frame 4/12 (drag)


...

  frame 8/12 (zoom_out)


  frame 9/12 (zoom_out)


  frame 10/12 (zoom_out)


  frame 11/12 (zoom_in)


  frame 12/12 (zoom_in)

12 frames capturados


In [3]:
if frames:
    frames[0].save(
        str(GIF_PATH),
        save_all=True,
        append_images=frames[1:],
        duration=900,
        loop=0,
        optimize=True,
    )
    print(f"GIF guardado: {GIF_PATH} ({GIF_PATH.stat().st_size/1024:.1f} KB)")
else:
    print("No se generaron frames")

GIF guardado: /home/bellic12/Desktop/Visual/Semana_13_SLAM_Robotica_Visual/semana_13_2_mapas_interactivos_datos_satelitales/media/05_mapa_interactivo.gif (3063.6 KB)


## Conclusiones

- El PNG estático muestra el estado inicial del mapa: NDVI, capas y controles.
- El GIF animado evidencia la interactividad: zoom, paneos suaves y la fluidez del Leaflet sobre los datos.
- Si en algún caso el `flyTo` no se ejecuta (la variable del mapa puede tener otro nombre según versión de folium), los frames aún muestran el render final del mapa, que sigue siendo demostrativo.